In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.14.1+cu130
True


In [2]:
torch.manual_seed(42)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
from torchvision import datasets
from torchvision.transforms import ToTensor

train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

print(len(train_data))
print(len(test_data))

60000
10000


In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True)

In [6]:
image, label = train_data[0]
print(image.shape, label, train_data.classes[label])

torch.Size([1, 28, 28]) 9 Ankle boot


In [7]:
from torch import nn

class FashionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = FashionModel().to(device)
print(model)

FashionModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [8]:
loss_fn = nn. CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
def train_one_round(loader, model, loss_fn, optimizer):
  model.train()
  for images, labels in loader:
      images, labels = images.to(device), labels.to(device)
      predictions = model(images)
      loss = loss_fn(predictions, labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
  return loss.item()

In [10]:
def test (loader, model):
  model.eval()
  correct = 0
  with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        predictions = model(images)
        correct += (predictions.argmax(1) == labels).sum().item()
  return correct / len(loader.dataset)

In [11]:
for epoch in range(5):
    last_loss = train_one_round(train_loader, model, loss_fn, optimizer)
    accuracy = test(test_loader, model)
    print(f"Round {epoch+1}: loss {last_loss:.3f}, test accuracy {accuracy:.1%}")

Round 1: loss 0.465, test accuracy 84.9%
Round 2: loss 0.324, test accuracy 86.3%
Round 3: loss 0.410, test accuracy 86.6%
Round 4: loss 0.114, test accuracy 87.5%
Round 5: loss 0.177, test accuracy 87.9%


In [12]:
correct_by_class = [0] * 10
total_by_class = [0] * 10

model.eval()
with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    guesses = model(images).argmax(1)
    for guess, label in zip (guesses, labels):
      total_by_class[label.item()] += 1
      if guess == label:
        correct_by_class[label.item()] += 1

for i, name in enumerate(train_data.classes):
  print(f"{name:12} {correct_by_class[i] / total_by_class[i]:.1%}")


T-shirt/top  83.1%
Trouser      96.0%
Pullover     85.3%
Dress        90.8%
Coat         72.6%
Sandal       96.7%
Shirt        66.3%
Sneaker      94.6%
Bag          98.4%
Ankle boot   95.5%


In [13]:
torch.save(model.state_dict(), "fashion_model.pth")